# Cross-source contradictions

Replaces `scripts/detect_conflicts.py`.

Four sources with incompatible models will disagree. Having four sources is not the same as reconciling four sources — and that was the reviewer's actual point about *problems a practitioner faces*.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Detect

Both sides of every contradiction are retained.

In [2]:
from biomedical.graph.store import GraphClient
from biomedical.knowledge import conflicts as cf
cfg = config.load()

with GraphClient(uri=cfg.neo4j.uri, username=cfg.neo4j.username,
                 password=cfg.neo4j.password.reveal(), database=cfg.neo4j.database) as c:
    indication = cf.detect_indication_vs_adverse_event(c)
    ambiguous = cf.detect_ambiguous_mentions(c, threshold=0.6)
    cross = cf.detect_cross_authority_identity(c)

found = indication + ambiguous + cross
print(cf.summarise(found))

{'total': 39, 'by_kind': {'indication_vs_adverse_event': 4, 'ambiguous_mention': 35}, 'requires_human': 39, 'auto_resolvable': 0, 'resolved': 0}


## Confounding by indication — the interesting one

Read naively, the FAERS edge says the drug caused the disease. But dacarbazine *is* a melanoma chemotherapy — the disease is in the reports because the patients have it.

FAERS has no denominator, no control group and heavy reporting bias. **Co-occurrence is not evidence of causation.**

In [3]:
for con in indication:
    print(con.describe())
    print()

[indication_vs_adverse_event] dacarbazine.relationship_to::Malignant melanoma  UNRESOLVED - requires a human
    clinicaltrials.gov@v2: indication: the drug is investigated FOR 'Unresectable Melanoma'
    openfda.faers@faers: co-reported WITH reaction 'Malignant melanoma'
    note: Probable confounding by indication. FAERS has no denominator and no control group, so co-occurrence is not evidence of causation. Source precedence cannot settle this — it needs clinical review.

[indication_vs_adverse_event] dacarbazine.relationship_to::Malignant melanoma  UNRESOLVED - requires a human
    clinicaltrials.gov@v2: indication: the drug is investigated FOR 'melanoma'
    openfda.faers@faers: co-reported WITH reaction 'Malignant melanoma'
    note: Probable confounding by indication. FAERS has no denominator and no control group, so co-occurrence is not evidence of causation. Source precedence cannot settle this — it needs clinical review.

[indication_vs_adverse_event] dacarbazine.relationship_

## Why this cannot be auto-resolved

Deciding whether a disease is the indication or the adverse event is a **clinical judgment**, not a ranking of sources. Source precedence would produce a confident wrong answer, so the detector refuses to try.

In [4]:
for kind in cf.ConflictKind:
    print(f"  {kind.value:<34} auto_resolvable={kind.auto_resolvable}")

from biomedical.foundation import AuthorityPrecedence
p = AuthorityPrecedence.default()
print("\ndeclared precedence:", p.order)
print("  prefer(openfda, rxnorm) =", p.prefer(("openfda", "rxnorm")))
print("  a tie returns None, so a human decides:", p.prefer(("unlisted_a", "unlisted_b")))

  indication_vs_adverse_event        auto_resolvable=False
  ambiguous_mention                  auto_resolvable=False
  cross_authority_identity           auto_resolvable=True
  property_disagreement              auto_resolvable=True

declared precedence: ('rxnorm', 'hgnc', 'mondo', 'clinicaltrials', 'openfda', 'chembl', 'catalog')
  prefer(openfda, rxnorm) = rxnorm
  a tie returns None, so a human decides: None


## Ambiguous mentions

Every candidate is retained; none is silently collapsed.

In [5]:
for con in ambiguous[:5]:
    print(f"[{con.kind.value}] {con.subject}")
    for s in con.sides:
        print(f"    {s.claim[:56]:<56} {s.evidence}")
    print()
print(f"... {len(ambiguous)} ambiguous mentions in total")

[ambiguous_mention] Lymphoproliferative Disorders
    lymphoproliferative syndrome (MONDO:0016537)             similarity 0.98
    methotrexate-associated lymphoproliferative disorders (M similarity 0.85
    atypical lymphoproliferative disorder (MONDO:0044921)    similarity 0.85
    chronic lymphoproliferative disorder of NK-cells (MONDO: similarity 0.78

[ambiguous_mention] Neuroendocrine Neoplasms
    anal canal neuroendocrine neoplasm (MONDO:0003504)       similarity 0.88
    colon neuroendocrine neoplasm (MONDO:0002882)            similarity 0.87
    rectum neuroendocrine neoplasm (MONDO:0003646)           similarity 0.85
    gastric neuroendocrine neoplasm (MONDO:0003111)          similarity 0.84

[ambiguous_mention] Ovarian Neoplasms
    ovarian cancer (MONDO:0008170)                           similarity 0.97
    ovarian neoplasm (MONDO:0021068)                         similarity 0.97
    ovarian yolk sac tumor (MONDO:0006344)                   similarity 0.76
    ovarian mucino

## Persisted state

A fact under unresolved contradiction defaults to **ineligible** as a rule input — so these are excluded three times over: low confidence, needs review, and contested.

In [6]:
from biomedical.ui import data
d = data.conflicts()
print(f"conflicts {d['total']}   unresolved {d['unresolved']}"
      f"   contested facts {d['contested_resolutions']}")
print()
for k in d["by_kind"]:
    print(f"  {k['kind']:<34} {k['count']}")

conflicts 37   unresolved 37   contested facts 36

  ambiguous_mention                  35
  indication_vs_adverse_event        2


## Four resolver bugs this surfaced

Looking for disagreement found defects nothing else had:

| Pattern | Before | After |
|---|---|---|
| subtypes outrank the general concept | `Inflammatory Bowel Disease` → `inflammatory bowel disease 13` *(a genetic locus)* | → the general term |
| hyphen variance costs similarity | MONDO `B-cell` vs trial `B cell` | `non-small cell lung cancer` → 1.000 |
| **search recall, not scoring** | the correct DLBCL term was not in OLS4's top 8 at all | window raised to 25 |
| leading acronym gloss | `IBD - ...` → locus 13 at 0.30 | → MONDO:0005265 at 0.95 |

The third is worth separating: I nearly tuned the scorer. It was a **recall** problem — the right answer was outside the result window, so no scoring change could have reached it.

In [7]:
from biomedical.sources.mondo import MondoTerm, best_similarity, search
from biomedical.sources.resolver import normalise_disease, resolve_disease

general = MondoTerm("MONDO:0005265", "inflammatory bowel disease")
locus = MondoTerm("MONDO:0012831", "inflammatory bowel disease 13")
q = "inflammatory bowel disease"
print(f"general {best_similarity(q, general):.3f}  vs  locus {best_similarity(q, locus):.3f}")

t = MondoTerm("MONDO:0018905", "diffuse large B-cell lymphoma")
print(f"hyphen variance: {best_similarity('diffuse large B cell lymphoma', t):.3f}")

print("\nsearch recall — the general DLBCL term by window size:")
for rows in (8, 25):
    hits = search("diffuse large B cell lymphoma", rows=rows)
    present = any(t.label.lower().replace("-", " ") == "diffuse large b cell lymphoma"
                  for t, _ in hits)
    print(f"  rows={rows:<3} general term present: {present}")

print("\nleading acronym gloss:")
print("  ", normalise_disease("IBD - Inflammatory Bowel Disease"))
print("  ", resolve_disease("IBD - Inflammatory Bowel Disease").describe())

general 1.000  vs  locus 0.946
hyphen variance: 1.000

search recall — the general DLBCL term by window size:


  rows=8   general term present: False


  rows=25  general term present: True

leading acronym gloss:
   inflammatory bowel disease


   IBD - Inflammatory Bowel Disease         -> inflammatory bowel disease (MONDO:0005265) 0.95 via normalised_exact
